# Data imputation techniques
Data processing is one of the most time consuming steps in the ML workflow, sometimes data collection pipelines treats unexpected values and data engineers don't notice to these issues. Also on some scenarios is regard to input errors or the freak is something related with bussines logic.  
So, as part of the preprocessing step before feeding data to our models, we have to decide "What am I going to do with missing data in my dataset?". This notebook tries to explain common techniques to face this issue explaining pros and counters. We are going to use synthetic dataset to compare the result of each technique

In [1]:
from pandas import read_csv
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

DATA = read_csv("/kaggle/input/datasets/sridevilavanyacse/student-lifestyle-and-stress-prediction-dataset/student-lifestyle-and-stress-dataset.csv")

CATS = ["Student_Type"]
NUMS = ["Sleep_Hours", "Study_Hours", "Social_Media_Hours", "Attendance", "Exam_Pressure", "Family_Support", "Month"]
TARGET = "Stress_Level"

data = DATA.copy()

def pipeline_training(imputer_cat, imputer_num, drop_data = False):
    if not drop_data:
        data_copy = data
        CAT_PIPELINE = Pipeline([
            ("encoding", OrdinalEncoder()),
            ("impute", imputer_cat)
        ])
        
        NUM_PIPELINE = Pipeline(steps=[
            ("impute", imputer_num),
            ("normalizing", StandardScaler())
        ])
        
        TRANSFORMER = ColumnTransformer([
            ("cat_data", CAT_PIPELINE, CATS),
            ("num_data", NUM_PIPELINE, NUMS),
        ])
    else:
        data_copy = data.dropna()
        print("Instances before dropping: ", len(DATA))
        print("Instances after dropping: ", len(data_copy))

        TRANSFORMER = ColumnTransformer([
            ("normalize", StandardScaler(), NUMS),
            ("encode", OrdinalEncoder(), CATS)
        ])
    
    x_train, x_test, y_train, y_test = train_test_split(data_copy[CATS + NUMS], data_copy[TARGET], test_size=0.2, random_state=123)
    
    X_train = TRANSFORMER.fit_transform(x_train)
    X_test = TRANSFORMER.transform(x_test)
    
    MODEL = XGBClassifier(random_state=123, learning_rate=5e-2, booster="gbtree", sampling_method="gradient_based")
    MODEL.fit(X_train, y_train)
    
    y_pred = MODEL.predict(X_test)
    ACCURACY = accuracy_score(y_test, y_pred)
    PRECISION = precision_score(y_test, y_pred)
    F1_SCORE = f1_score(y_test, y_pred)
    RECALL = recall_score(y_test, y_pred)
    
    print("----------------------------")
    print(f"Accuracy: {ACCURACY:0.2f}")
    print(f"Precision: {PRECISION:0.2f}")
    print(f"Recall: {RECALL:0.2f}")
    print(f"F1-Score: {F1_SCORE:0.2f}")

    return { "accuracy": ACCURACY, "precision": PRECISION, "f1-score": F1_SCORE, "recall": RECALL}

## 1. Discard missing data
This a basic strategy that consists on dropping row instances or columns with null values. However, the price is lossing valuable information that could be significant to your model. Also if your dataset is small is possible to your models will suffer overfitting.

In [2]:
METRICS_DROPPED_DATA = pipeline_training(None, None, True)

Instances before dropping:  25500
Instances after dropping:  16809
----------------------------
Accuracy: 0.81
Precision: 0.75
Recall: 0.57
F1-Score: 0.65


## 2. Univariate feature imputation
This category bounds methods ignoring feature relationships, are simple and computationally efficient.

1. **Numerical features:** Calculates the median or mean of the column and imputes missing values with it. Median works fine for skewed and mean is better for normally distributed data.
2. **Categorical features:** Performs a mode calculation for each columns, in case of a column has more than 1 mode, the first is selected.
3. **Other approach:** As the other, consist on manually selecting a constant value to impute in each column.

This technique is recommended for small amounts of missing data (almost 5% of the dataset size) because variable distribution doesn't suffers a lot.

In [3]:
METRICS_UNIVARIATE = pipeline_training(
    SimpleImputer(strategy="most_frequent"),
    SimpleImputer(strategy="mean"),
    False
)

----------------------------
Accuracy: 0.82
Precision: 0.75
Recall: 0.61
F1-Score: 0.67


## 3. Multivariate feature imputation
These approaches are more complex than univariate methods, so it allows to hold feature relationships. Every strategy is based on estimating a feature value from the other fields.

### KNN imputation
It consists on training a KNN model that will estimate feature values from the k-nearest neighbors to data instance. Final estimation is a weighted mean from de values of the closest instances. KNN imputation is fine because holds underlying patterns and relationships among variables but is too slow in big datasets.

In [4]:
METRICS_KNN = pipeline_training(
    SimpleImputer(strategy="most_frequent"),
    KNNImputer(),
    False
)

----------------------------
Accuracy: 0.82
Precision: 0.74
Recall: 0.60
F1-Score: 0.67


### Linear models (logistic and linear regression)
So, this is simple as training a linear (logistic for categorical features) regression model to estimate a feature value. Linear models are simple and relatively quick trainable, but have as the name sugests, works well when relationships among variables are linear. On data that has a quadratic relationship for instance, linear models never will be able to impute values on a good way.  
An additional care must be taken when working with numeric variables because you must be careful if the model estimates a negative value on a possitive variable. This approach requires training a linear estimator per variable or if features with null values has a similar distribution, a single multioutput estimator is enough.

### Multiple imputation by chained equations (MICE)
MICE is a very sophisticated but powerful algorithm to address data imputation on multivariate scenarios. It works treating each feature with missing values a target variable and uses other features to predict those missing values, at the begin assings an initial value (usually variable mean o median) to make predictions, when an iteration is completed (all variables are imputed once) the process is repeated until reach a stable state. This approach holds relationships among variables at imputed data, suites fine for complex data and works for numerical and categorical data. As you guess, its main disadvantage comes from his benefits, is a computationally intensive algorithm.

In [5]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression, LinearRegression

# By default, uses a BayesianRidge model
# In this case we are using linear models
METRICS_MICE = pipeline_training(
    IterativeImputer(
        LogisticRegression(random_state=123)
    ),
    IterativeImputer(
        LinearRegression()
    ),
    False
)

----------------------------
Accuracy: 0.82
Precision: 0.75
Recall: 0.61
F1-Score: 0.67
